# 🎓 AWREAS Direct Benchmark Evaluator (No-Ngrok / Local CPU Setup)

This notebook allows you to run the complete **Academic Writing Review and Evaluation Agentic System (AWREAS)** evaluation pipeline directly in Google Colab's Python memory without spawning Uvicorn, dealing with Ngrok tunnels, or hitting external API limits.

It is fully optimized to run the **Rule-Based Engine** extremely fast and requires zero network requests.

## 🛠️ Step 1: Mount Google Drive & Unzip Codebase

Upload your `academic-review-system.zip` file directly to the root of your Google Drive (`My Drive/`), then run this cell to mount Drive and extract the project directly into Colab's ultra-fast local runtime (`/content/`).

*(This cell automatically searches the zip archive and changes the working directory to the correct folder containing `pyproject.toml` no matter how it was zipped).*

In [ ]:
from google.colab import drive
import os
import sys
import glob

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define zip path and target extraction directory
zip_path = '/content/drive/MyDrive/academic-review-system.zip'
extract_dir = '/content/academic-review-system'

if os.path.exists(zip_path):
    print("Found zip file! Extracting to local Colab storage...")
    !unzip -q -o {zip_path} -d {extract_dir}
    print("Extraction complete!")
else:
    # Fallback if they uploaded the unzipped directory to Drive
    print("Zip file not found at root. Checking for folder instead...")
    extract_dir = '/content/drive/MyDrive/academic-review-system'
    if not os.path.exists(extract_dir):
        raise FileNotFoundError("Please make sure your project is uploaded to Google Drive!")

# 3. Dynamically find the folder containing pyproject.toml (handles any nested folder layout)
toml_paths = glob.glob(os.path.join(extract_dir, '**/pyproject.toml'), recursive=True)

if toml_paths:
    project_root = os.path.dirname(toml_paths[0])
    print(f"\n🎯 Success! Found pyproject.toml at: {project_root}")
    sys.path.append(project_root)
    %cd {project_root}
else:
    raise FileNotFoundError("ERROR: Could not locate 'pyproject.toml' anywhere inside the extracted files. Please make sure you zipped the correct project.")

## 📦 Step 2: Install Project Dependencies

Since this project uses a modern package structure defined in `pyproject.toml`, we can install the package and its requirements directly using `pip install .`:

In [ ]:
# Install core dependencies directly from pyproject.toml
!pip install -q .

# Optional: If you need additional ML packages (spacy, sentence-transformers, etc.):
# !pip install -q ".[ml]"

## 📊 Step 3: Stage 1 - Dataset Preparation

Extract the 50 persuasive student essays (Prompt 1) and their gold-standard human rater scores from the raw ASAP dataset.

In [ ]:
!python scripts/prepare_asap_benchmark.py

## ⚡ Step 4: Stage 2 & 3 - Run Direct Rule-Based Evaluation

Instead of setting up HTTP servers and client-side POST requests, we import `SupervisorAgent` directly into memory. 

By passing `llm_client=None`, the supervisor automatically runs all specialist workers in **rule-based mode**. We also explicitly set `settings.ENABLE_WEB_RESEARCH = False` to disable external network lookups (DuckDuckGo/CrossRef) for each essay, ensuring the benchmark completes in a few seconds.

In [ ]:
import json
import os

# Import settings and disable external web research lookups for ultra-fast runs
from app.core.config import settings
settings.ENABLE_WEB_RESEARCH = False

# Import SupervisorAgent directly
from app.supervisor.agent import SupervisorAgent

input_file = "data/asap/benchmark_prompt1.jsonl"
output_file = "data/asap/benchmark_results_prompt1.json"

# Read the dataset and pre-calculate length to avoid hardcoded print indicators
with open(input_file, "r") as f:
    dataset_lines = [json.loads(line) for line in f if line.strip()]

total_essays = len(dataset_lines)
results = []

print("Initializing SupervisorAgent in Rule-Based Mode...")
supervisor = SupervisorAgent(llm_client=None)

print(f"\nStarting direct Rule-Based evaluation on {total_essays} essays...")

for i, essay_data in enumerate(dataset_lines):
    essay_text = essay_data["content"]
    essay_id = essay_data["id"]
    
    # Execute the evaluator directly in Colab CPU memory
    # Colab supports top-level await natively!
    import time
    start_time = time.perf_counter()
    context = await supervisor.run_evaluation(essay_text)
    processing_time_ms = (time.perf_counter() - start_time) * 1000
    
    # Format results exactly how compute_agreement.py expects them
    results.append({
        "system": "AWREAS_Full",
        "case_id": str(essay_id),
        "ok": True,
        "overall_score": context.overall_score,
    })
    
    print(f"Processed essay {i+1}/{total_essays} (ID: {essay_id}) -> Score: {context.overall_score}")

# Wrap under 'results' root key to match the evaluation client output schema
output_data = {
    "results": results
}

# Save the benchmark results locally
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w") as out:
    json.dump(output_data, out, indent=4)

print(f"\n🎉 Success! Evaluation complete. Saved {len(results)} records to: {output_file}")

## 📈 Step 5: Stage 4 - Compute Statistical Agreement & Chapter 4 Metrics

Run this cell to compute **Quadratic Weighted Kappa (QWK)**, **Mean Absolute Error (MAE)**, **Pearson Correlation (r)**, and the rest of your statistics for Chapter 4.

In [ ]:
!python scripts/compute_agreement.py \
    --results data/asap/benchmark_results_prompt1.json \
    --gold data/asap/gold_prompt1.json

## 💾 Step 6: Backup Results back to Google Drive

If you extracted the zip locally inside the Colab instance (`/content/`), copy your generated benchmark results back to your mounted Google Drive so you don't lose them when the Colab runtime is closed.

In [ ]:
import shutil

drive_dest = '/content/drive/MyDrive/AWREAS_Benchmark_Results'
os.makedirs(drive_dest, exist_ok=True)

shutil.copy('data/asap/benchmark_results_prompt1.json', drive_dest)
print(f"Successfully backed up results to Google Drive folder: {drive_dest}")